# Machine Learning
## Programming Assessment 5: Neural Networks

### Instructions


*   The aim of this assignment is to learn machine learning tools - Keras, Sklearn and PyTorch.
*   You must use the Python programming language.
*   You can add as many code/markdown cells as required.
*   ALL cells must be run (and outputs visible) in order to get credit for your work.
*   Please use procedural programming style and comment your code thoroughly.
*   There are three parts of this assignment. The import statements for the required libraries is already given.


### Introduction
In this assignment, you will be using neural networks to implement a simplified version of a speech recognizer which aims to identify what digit has been spoken in a given audio file.

In order to accomplish this, you will be using different toolkits, popularly used in machine learning for training models. In this assignment, you will be introduced to Sklearn, Keras, and Pytorch. An implementation from scratch is not required for the purposes of this assignment.

Have fun!

In [2]:
import numpy as np
import pandas as pd

## Part 1: Feature Extraction
You will the MNIST audio dataset which can be downloaded from [here](https://www.kaggle.com/datasets/sripaadsrinivasan/audio-mnist). The dataset contains audio recordings, where speakers say digits (0 to 9) out loud. Use the following line of code to read the audio file:
```python
audio, sr = librosa.load(file_path, sr=16000)
```
You need to extract MFCC features for each audio file, the feature extraction code is give (you can read about MFCC from [here](https://link.springer.com/content/pdf/bbm:978-3-319-49220-9/1.pdf)). Length of each feature vector will be 13. You need to save all the feature vectors in a csv file with ith column representing ith feature, and each row representing an audio file. Add a 'y' column to the csv file and append the labels column at the end. Your csv file should look like this:

| x1 | x2 | x3 | x4 | x5 | x6 | x7 | x8 | x9 | x10 | x11 | x12 | x13 | y |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| -11.347038 | -8.070062 | -0.915299 | 6.859546 | 8.754656 | -3.440287 | -5.738487 | -21.853178 | -9.859462 | 3.584948 | -2.661195	| 1.023747 | -4.574332 | 2 |

Print out 2 vectors in this notebook.

Split the dataset into train and test with 80:20 ratio. Print the train data size and test data size.

In [3]:
from glob import glob
import python_speech_features as mfcc
import librosa
from sklearn.model_selection import train_test_split

In [49]:
from IPython.display import Audio
import kagglehub
import os

# Download latest version
#file_path = kagglehub.dataset_download("sripaadsrinivasan/audio-mnist")
file_path = kagglehub.dataset_download("sripaadsrinivasan/audio-mnist")
print("Path to dataset files:", file_path)

Path to dataset files: C:\Users\LP\.cache\kagglehub\datasets\sripaadsrinivasan\audio-mnist\versions\1


In [5]:
data_path = os.path.join(file_path, "data")
# Automatically find any file)
audio_file = None

for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".wav"):
            audio_file = os.path.join(root, file)
            break	
    if audio_file:
        break
print("Selected file: ", audio_file)    

# Now loading the audio file
audio, sr = librosa.load(audio_file, sr=16000)
Audio(audio, rate=16000)

#audio_file = os.path.join()
#audio, sr = librosa.load(file_path, sr=16000)
#Audio(audio, rate=16000)

Selected file:  C:\Users\LP\.cache\kagglehub\datasets\sripaadsrinivasan\audio-mnist\versions\1\data\01\0_01_0.wav


In [6]:
def get_MFCC(audio, sr):
    features = mfcc.mfcc(audio, sr, 0.025, 0.01, 13, appendEnergy = True)
    return np.mean(features, axis=0)

In [51]:
import os
data_dir = data_path
file_paths = glob(os.path.join(data_dir, "*/*.wav"))
data = []
labels = []
print("Files Records: ",len(file_paths))


Files Records:  30000


In [8]:
data = []
labels = []

from tqdm import tqdm
for file_path in tqdm(file_paths):
    
    label = int(os.path.basename(file_path).split('_')[0])
    
    audio, sr = librosa.load(file_path, sr=16000)
    
    features = get_MFCC(audio, sr)
    
    data.append(features)
    labels.append(label)
    
data = np.array(data)
labels = np.array(labels)
df = pd.DataFrame(data, columns=[f"x{i+1}" for i in range(13)])
df["y"] = labels
df.to_csv("data.csv", index=False)

print("Saved:", df.shape)


100%|██████████| 30000/30000 [07:43<00:00, 64.76it/s]


Saved: (30000, 14)


In [9]:
print(features.shape)

(13,)


In [10]:
df.head()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,y
0,-10.666645,-1.411540,-2.473520,7.168286,-3.494081,-2.785832,-14.398475,-4.217254,3.395590,-11.173119,-2.370115,3.899111,-11.044119,0
1,-11.414706,-2.149634,-0.706419,10.050852,1.945194,-5.537895,-12.362039,0.239243,1.692732,-6.133325,3.241282,5.820779,-7.441492,0
2,-10.242019,-2.336824,-5.271823,5.837609,-1.337890,-2.493693,-13.572080,-7.714594,0.306567,-6.591890,-1.882395,3.712337,-6.204464,0
3,-10.193638,-3.643128,-8.129836,2.650802,-0.697407,-8.648287,-18.195700,-2.643146,9.175978,-4.324699,-1.795747,7.319704,-11.465352,0
4,-10.485194,-4.505045,-0.209834,2.821504,-3.985702,-4.391896,-15.959238,-1.718976,2.344726,-8.538168,-1.725910,4.176098,-7.359689,0


In [11]:
X_train_eval, X_test, y_train_eval, y_test = train_test_split(np.array(data), labels, test_size = 0.2, random_state = 42, stratify = labels)

print("Train Size: ", X_train_eval.shape)
print("Test Size: ", X_test.shape)

Train Size:  (24000, 13)
Test Size:  (6000, 13)


## Part 2: Neural Network Implementation

### Task 2.1:  Scikit-learn

In this part you will use the [Scikit-learn](https://scikit-learn.org/stable/index.html) to implement the [Neural Network](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) and apply it to the MNIST audio dataset (provided in part 1). Split the training dataset into train and evaluation data with 90:10 ratio. Run evaluation on X_eval while training on X_train. Tune the hyperparameters to get the best possible classification accuracy. You need to report accuracy, recall, precision and F1 score on the test dataset and print the confusion matrix.

Expected value for accuracy is 87 or above.

In [12]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [13]:
# Evaluation Data Split 90:10
X_train, X_eval, y_train, y_eval = train_test_split(X_train_eval, y_train_eval, test_size=0.10, random_state=42, stratify = y_train_eval)

print("Train Size: ", X_train.shape)
print("Evaluation Size: ", X_eval.shape)
print("Test Size: ", y_test.shape)

mlp = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=800, random_state = 42)

mlp.fit(X_train, y_train)

y_pred_eval = mlp.predict(X_eval)

print("Scikit-learn MLP:")
print(classification_report(y_eval, y_pred_eval))
print(confusion_matrix(y_eval, y_pred_eval))


Train Size:  (21600, 13)
Evaluation Size:  (2400, 13)
Test Size:  (6000,)
Scikit-learn MLP:
              precision    recall  f1-score   support

           0       0.91      0.90      0.91       240
           1       0.98      0.95      0.97       240
           2       0.91      0.93      0.92       240
           3       0.94      0.94      0.94       240
           4       0.98      0.99      0.98       240
           5       0.97      0.97      0.97       240
           6       0.99      1.00      0.99       240
           7       0.98      0.97      0.97       240
           8       0.97      0.96      0.97       240
           9       0.97      0.97      0.97       240

    accuracy                           0.96      2400
   macro avg       0.96      0.96      0.96      2400
weighted avg       0.96      0.96      0.96      2400

[[217   0  15   2   2   1   0   2   1   0]
 [  1 229   2   0   2   0   0   0   0   6]
 [ 11   1 222   5   0   0   0   0   0   1]
 [  3   0   6 226   

### Task 2.2: Tensorflow Keras

In this part you will use the [Keras](https://www.tensorflow.org/api_docs/python/tf/keras/Sequential) to implement the [Neural Network](https://machinelearningmastery.com/tutorial-first-neural-network-python-keras/) and apply it to the MNIST audio dataset (provided in part 1). Split the training dataset into train and evaluation data with 90:10 ratio. Run evaluation on X_eval while training on X_train. Tune the hyperparameters to get the best possible classification accuracy. You need to report accuracy, recall, precision and F1 score on the test dataset and print the confusion matrix.

Expected value for accuracy is 87 or above.

In [18]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import optimizers

print("Tensorflow imported successfuly. version: ", tf.__version__)

Tensorflow imported successfuly. version:  2.20.0


In [19]:
from tensorflow.keras.optimizers import Adam

# Set the parameters accordingly
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
EPOCHS = 100

In [20]:

model = Sequential([
	Dense(128, activation = 'relu', input_shape = (13,)),
	Dense(64, activation = 'relu'),
    Dense(10, activation = 'softmax')
])

model.compile(optimizer = Adam(learning_rate = LEARNING_RATE), loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

model.fit(X_train, y_train, validation_data=(X_eval, y_eval), epochs = EPOCHS, batch_size = BATCH_SIZE)

y_pred_eval = np.argmax(model.predict(X_eval), axis = 1)

print(" Keras MLP")
print(classification_report(y_eval, y_pred_eval))
print(confusion_matrix(y_eval, y_pred_eval))



Epoch 1/100


c:\Users\LP\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7804 - loss: 0.6606 - val_accuracy: 0.8650 - val_loss: 0.3474
Epoch 2/100
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8808 - loss: 0.3183 - val_accuracy: 0.8967 - val_loss: 0.2786
Epoch 3/100
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9013 - loss: 0.2659 - val_accuracy: 0.9079 - val_loss: 0.2524
Epoch 4/100
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9144 - loss: 0.2341 - val_accuracy: 0.9125 - val_loss: 0.2391
Epoch 5/100
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9217 - loss: 0.2150 - val_accuracy: 0.9233 - val_loss: 0.2116
Epoch 6/100
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9294 - loss: 0.1956 - val_accuracy: 0.9096 - val_loss: 0.2357
Epoch 7/100
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9315 - loss: 0.1832 - val_accuracy: 0.9204 - val_loss: 0.1988
Epoch 8/100
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9381 - loss: 0.1713 - val_accuracy: 0.9287

### Task 2.3: Pytorch

In this part you will use the [Keras](https://pytorch.org/docs/stable/nn.html) to implement the [Neural Network](https://medium.com/analytics-vidhya/a-simple-neural-network-classifier-using-pytorch-from-scratch-7ebb477422d2) and apply it to the MNIST audio dataset (provided in part 1). Split the training dataset into train and evaluation data with 90:10 ratio. Run evaluation on X_eval while training on X_train. You need to use DataLoader to generate batches of data. Tune the hyperparameters to get the best possible classification accuracy. You need to report training loss, training accuracy, validation loss and validation accuracy after each epoch in the following format:
```
Epoch 1/2
loss: 78.67749792151153 - accuracy: 0.6759259259259259 - val_loss: 6.320814955048263 - val_accuracy: 0.7356481481481482
Epoch 2/2
loss: 48.70551285566762 - accuracy: 0.7901234567901234 - val_loss: 6.073690168559551 - val_accuracy: 0.7791666666666667
```
You need to report accuracy, recall, precision and F1 score on the test dataset and print the confusion matrix.

Expected value for accuracy is 87 or above.

In [22]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

In [23]:
class Data(Dataset):
    def __init__(self, X, y):
        # Code here
        self.X = torch.tensor(X, dtype = torch.float32)
        self.y = torch.tensor(y, dtype = torch.long)
        #pass
    def __getitem__(self, index):
        
        # Code here
        return self.X[index], self.y[index]
        

    def __len__(self):
        # Code here
        return len(self.X)
        
        # Code here

In [30]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        # Code here
        # define layers
        self.model = nn.Sequential(
            nn.Linear(13, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64,10)
        )
    
    def forward(self, x):
        # Code here
        return self.model(x)

In [25]:
# Set the parameters accordingly
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
EPOCHS = 50

In [ ]:
# Set the loss function and optimizer accordingly
import torch.optim as optim
model = NeuralNetwork()
loss_function = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [33]:
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

In [38]:
train_loader = DataLoader(Data(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

eval_loader = DataLoader(Data(X_eval, y_eval), batch_size=BATCH_SIZE, shuffle=False)

test_loader = DataLoader(Data(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

for epoch in range(EPOCHS):
    
    
    # TRAIN
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for x_batch, y_batch in train_loader:
        
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        
        logots = model(x_batch)
        
        loss = loss_function(logots, y_batch)
        
        loss.backward()
        
        optimizer.step()
        
        total_loss += loss.item() * x_batch.size(0)
        
        preds = torch.argmax(logots, dim=1)
        correct += torch.sum(preds == y_batch)
        total += y_batch.size(0)
        
    train_loss = total_loss / total
    train_acc = correct / total
    
    # EVAL
    
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        
        for x_batch, y_batch in eval_loader:
            
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits = model(x_batch)
            
            loss = loss_function(logits, y_batch)
            
            val_loss += loss.item() * x_batch.size(0)
            
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += y_batch.size(0)
        
        eval_loss = val_loss / val_total
        eval_acc = val_correct / val_total
        
        print(
            f"Epoch: {epoch + 1}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Eval Loss: {eval_loss:.4f}, Eval Acc: {eval_acc:.4f}"
        )
            
            

Epoch: 1, Train Loss: 0.2778, Train Acc: 0.8965, Eval Loss: 0.2526, Eval Acc: 0.9117
Epoch: 2, Train Loss: 0.2396, Train Acc: 0.9104, Eval Loss: 0.2380, Eval Acc: 0.9113
Epoch: 3, Train Loss: 0.2189, Train Acc: 0.9182, Eval Loss: 0.2199, Eval Acc: 0.9179
Epoch: 4, Train Loss: 0.1949, Train Acc: 0.9293, Eval Loss: 0.2260, Eval Acc: 0.9113
Epoch: 5, Train Loss: 0.1819, Train Acc: 0.9347, Eval Loss: 0.2189, Eval Acc: 0.9179
Epoch: 6, Train Loss: 0.1714, Train Acc: 0.9384, Eval Loss: 0.2026, Eval Acc: 0.9246
Epoch: 7, Train Loss: 0.1602, Train Acc: 0.9417, Eval Loss: 0.1872, Eval Acc: 0.9342
Epoch: 8, Train Loss: 0.1493, Train Acc: 0.9465, Eval Loss: 0.1934, Eval Acc: 0.9287
Epoch: 9, Train Loss: 0.1415, Train Acc: 0.9484, Eval Loss: 0.1867, Eval Acc: 0.9342
Epoch: 10, Train Loss: 0.1333, Train Acc: 0.9528, Eval Loss: 0.1736, Eval Acc: 0.9354
Epoch: 11, Train Loss: 0.1275, Train Acc: 0.9523, Eval Loss: 0.1926, Eval Acc: 0.9292
Epoch: 12, Train Loss: 0.1195, Train Acc: 0.9569, Eval Loss: 0.

In [48]:
# Final Test after training
model.eval()

test_loss = 0.0
test_correct = 0
test_total = 0

all_preds = []
all_true = []

with torch.no_grad():
    
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        logits = model(x_batch)
        
        loss = loss_function(logits, y_batch)
        test_loss += loss.item() * x_batch.size(0)
        
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        
        test_correct += (preds == y_batch).sum().item()
        test_total += x_batch.size(0)
        
        preds = torch.argmax(logits, dim=1)
        
        all_preds.append(preds.cpu().numpy())
        all_true.append(y_batch.cpu().numpy())

total_loss = test_loss / test_total  
test_acc = test_correct / test_total
      
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)

print("\nPyTorch MLP (TEST): ")

print("\nAccuracy: %.2f%%" % (test_acc * 100))

print("\nLoss: %.2f%%" % (total_loss))

print("\nClassification report:")
print(classification_report(y_true, y_pred))

print("confusion matrix:")
print(confusion_matrix(y_true, y_pred))


PyTorch MLP (TEST): 

Accuracy: 95.10%

Loss: 0.23%

Classification report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       600
           1       0.97      0.94      0.95       600
           2       0.93      0.88      0.90       600
           3       0.92      0.94      0.93       600
           4       0.97      0.97      0.97       600
           5       0.97      0.98      0.98       600
           6       0.99      0.99      0.99       600
           7       0.96      0.97      0.96       600
           8       0.95      0.97      0.96       600
           9       0.94      0.96      0.95       600

    accuracy                           0.95      6000
   macro avg       0.95      0.95      0.95      6000
weighted avg       0.95      0.95      0.95      6000

confusion matrix:
[[544   4  28   9   6   0   0   7   0   2]
 [  4 562   1   0   6   5   0   2   0  20]
 [ 36   1 527  23   0   0   0   7   3   3]
 [  7   0   7 565